# الدرس 07 - نمط تصميم التخطيط

يوضح هذا الدفتر نوتبوك **نمط تصميم التخطيط** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل Microsoft Agent.
ستتعلم كيفية تفكيك طلبات السفر المعقدة إلى مهام فرعية منظمة، وتكليفها إلى وكلاء متخصصين،
وتنفيذ الخطة الناتجة — كل ذلك باستخدام المخرجات المنظمة المدعومة بنماذج Pydantic.


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## تفكيك المهمة

تفكيك المهمة هو جوهر نمط التصميم التخطيطي. بدلاً من طلب وكيل واحد للتعامل مع طلب معقد من البداية إلى النهاية،
نقوم بتقسيم المشكلة إلى **مهام فرعية** أصغر معرّفة بشكل واضح.
يتم تعيين كل مهمة فرعية إلى وكيل متخصص (مثل رحلات الطيران، الفنادق، الأنشطة) مع أولويات واضحة
وترتيب تبعيات.

توفر هذه الطريقة عدة فوائد:
- **الوضوح**: كل مهمة فرعية لها مسؤولية واحدة.
- **التوازي**: يمكن تشغيل المهام الفرعية المستقلة بالتزامن.
- **الموثوقية**: يتم عزل الإخفاقات في المهام الفرعية الفردية.
- **تتبع الميزانية**: يتم تقدير التكاليف لكل مهمة فرعية ويتم تجميعها.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## إنشاء وكيل تخطيط مع إخراج منظم

يعمل وكيل التخطيط كـ **منسق استقبال**. عند تلقي طلب سفر على مستوى عالٍ،
يقوم بإنتاج `TravelPlan` منظم — يفكك الطلب إلى مهام فرعية، يحدد الأولويات،
ويحدد التبعيات بحيث يمكن للكونسيرج أو طبقة التنفيذ تنفيذ العمل.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## تنفيذ خطة بأدوات متخصصة

بمجرد أن يقوم موظف الاستقبال بإعداد خطة مُنظمة، يقوم **وكيل الكونسييرج** بتنفيذها.
تتعامل كل أداة متخصصة مع فئة واحدة من المهام الفرعية (رحلات جوية، فنادق، أنشطة). يقوم الكونسييرج
بالتكرار عبر المهام الفرعية للخطة حسب ترتيب التبعية ويُرسل كل واحدة إلى
الأداة المناسبة.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## ملخص

في هذا الدرس تعلمت **نمط تصميم التخطيط** لوكلاء الذكاء الاصطناعي:

1. **تفكيك المهمة** — يقوم وكيل التخطيط الأمامي بتقسيم طلب السفر المعقد إلى
   مهام فرعية منظمة باستخدام نماذج Pydantic، مخصصًا كل منها لوكيل متخصص مع الأولويات
   والاعتمادات.
2. **الإخراج المنظم** — من خلال تمرير `response_format` يعيد الوكيل كائن `TravelPlan` 
   مُحقق بدلاً من النص الحر، مما يجعل معالجة النتائج التالية موثوقة.
3. **تنفيذ الخطة** — يقوم وكيل الكونسيرج بالتكرار عبر المهام الفرعية باستخدام أدوات متخصصة
   (`book_flight`، `reserve_hotel`، `book_activity`) لتنفيذ الخطة والإبلاغ عن النتائج.

هذا النمط يفصل *ما يجب فعله* (التخطيط) عن *كيفية القيام به* (التنفيذ)، مما يجعل الوكلاء
أكثر تكاملية، قابلية للاختبار، وأسهل للتوسيع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
